### Instalação das bibliotecas

No terminal, instale as bibliotecas necessárias:

pip install pyodbc<br>
pip install sqlalchemy<br>
python -m pip install pyarrow<br>
python -m pip install azure-storage-file-datalake

No sql server abra-o e veja na tela Arquivo → Conectar Pesquisador de Objetos

Precisamos saber algo semelhante a:

Server name: DESKTOP-XXXX\SQLEXPRESS

Authentication: Windows Authentication

Database: nome_do_seu_banco

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

load_dotenv()


def conectar_sql_server():

    servidor = os.getenv("SQL_SERVER")
    banco = os.getenv("SQL_DATABASE")

    parametros = quote_plus(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={servidor};"
        f"DATABASE={banco};"
        f"Trusted_Connection=yes;"
    )

    engine = create_engine(
        f"mssql+pyodbc:///?odbc_connect={parametros}"
    )

    return engine

In [2]:
engine = conectar_sql_server()

print(engine)

Engine(mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3Dlocalhost%3BDATABASE%3Decommerce_bi%3BTrusted_Connection%3Dyes%3B)


In [3]:
df_clientes = pd.read_sql(
    "SELECT * FROM clientes",
    engine
)

df_clientes.head()

,id_cliente,nome,data_nascimento,cidade,estado,data_cadastro,status_cliente
0,1,Otávio Almeida Gomes,1956-06-20,São Paulo,SP,2024-11-24,ativo
1,2,Mariana Barbosa Freitas,1995-06-19,SANTOS,SP,2023-04-07,ativo
2,3,Elisa Moreira Ribeiro,1988-05-08,Londrina,PR,2024-10-18,ativo
3,4,Elisa Costa Teixeira,1972-08-14,São Paulo,SP,2024-04-06,ativo
4,5,Juliana Moreira Martins,2001-06-15,Rio de Janeiro,RJ,2024-09-29,inativo


In [4]:
print("Linhas:", df_clientes.shape[0])
print("Colunas:", df_clientes.shape[1])

Linhas: 10000
Colunas: 7


In [5]:
# Transformar df_clientes em Parquet na memória RAM.

from io import BytesIO

buffer = BytesIO()

df_clientes.to_parquet(
    buffer,
    index=False,
    engine="pyarrow"
)

buffer.seek(0)

print("DataFrame convertido para Parquet com sucesso!")

DataFrame convertido para Parquet com sucesso!


In [6]:
from azure.storage.filedatalake import DataLakeServiceClient

connection_string = os.getenv(
    "AZURE_STORAGE_CONNECTION_STRING"
)

datalake_service = DataLakeServiceClient.from_connection_string(
    connection_string
)

file_system = datalake_service.get_file_system_client(
    file_system="bronze"
)

In [7]:
diretorio = file_system.get_directory_client(
    "comercial"
)

In [8]:
arquivo = diretorio.get_file_client(
    "clientes.parquet"
)


In [9]:
arquivo.upload_data(
    buffer.getvalue(),
    overwrite=True
)

print("clientes.parquet enviado para Bronze com sucesso!")

clientes.parquet enviado para Bronze com sucesso!


In [10]:
tabelas = [
    "clientes",
    "pedidos",
    "itens_pedido",
    "pagamentos",
    "produtos"
]

In [11]:
from io import BytesIO

for tabela in tabelas:

    # Extrair do SQL Server
    df = pd.read_sql(
        f"SELECT * FROM {tabela}",
        engine
    )

    # Converter para Parquet em memória
    buffer = BytesIO()

    df.to_parquet(
        buffer,
        index=False,
        engine="pyarrow"
    )

    # Definir arquivo no Azure
    arquivo = diretorio.get_file_client(
        f"{tabela}.parquet"
    )

    # Enviar para Bronze
    arquivo.upload_data(
        buffer.getvalue(),
        overwrite=True
    )

    print(
        f"{tabela}: {len(df)} registros enviados para Bronze."
    )

clientes: 10000 registros enviados para Bronze.
pedidos: 50000 registros enviados para Bronze.
itens_pedido: 125000 registros enviados para Bronze.
pagamentos: 53000 registros enviados para Bronze.
produtos: 1000 registros enviados para Bronze.
